# Week 4: Code Transformation and Documentation Assistant

This project is a **two-tab Gradio application** that uses **LLMs to work with code** in practical and interactive ways.

## Tab 1: Python to C++ Converter
In this tab, the user can:
- paste Python code
- choose a provider and model
- convert the Python code into **optimized C++**
- run both versions side by side to compare their performance

This helps demonstrate how LLMs can assist with **code translation** and **basic performance-oriented experimentation**.

## Tab 2: Docstring Adder
In this tab, the user can:
- paste code in a selected programming language
- choose a model
- automatically add:
  - module-level docstrings
  - function docstrings
  - class docstrings
  - inline comments

The app improves code readability and documentation quality **without changing the original logic**.

## What This Project Demonstrates
- Gradio-based multi-tab application development
- LLM-powered code conversion
- LLM-assisted code documentation
- Model/provider selection for flexible experimentation
- Practical software engineering workflows using GenAI

## Import Required Libraries

In this cell, we import the libraries needed for the Week 4 Gradio application.

- `os` is used for environment variable handling and file-related operations.
- `io` is used for in-memory input/output handling.
- `sys` is used for interacting with the Python runtime environment.
- `subprocess` is used to execute external commands, such as compiling or running code.
- `load_dotenv` loads environment variables from a `.env` file.
- `OpenAI` is used to interact with OpenAI-compatible language models.
- `gradio` is used to build the interactive multi-tab web application.
- `InferenceClient` from `huggingface_hub` is used to access Hugging Face-hosted inference models.

These imports prepare the notebook for code conversion, documentation generation, model access, and UI development.

In [ ]:
# Standard library imports
import os
import io
import sys
import subprocess

# Third-party imports
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from huggingface_hub import InferenceClient

## Load Environment Variables and Check Provider Keys

In this cell, we load environment variables from the `.env` file and read API keys for multiple model providers.

The cell:
- loads keys for providers such as OpenAI, Anthropic, Google, Groq, Grok, OpenRouter, and Hugging Face
- cleans the Hugging Face token to remove unwanted spaces or surrounding quotes
- prints a basic status message showing whether each provider key is available

This setup makes the application flexible because it can work with multiple providers depending on which credentials are configured.

In [ ]:

# Load environment variables from the .env file
load_dotenv(override=True)

# Read API keys for different providers from environment variables
openai_api_key     = os.getenv("OPENAI_API_KEY")
ollama_api_key     = os.getenv("OLLAMA_API_KEY")
anthropic_api_key  = os.getenv("ANTHROPIC_API_KEY")
google_api_key     = os.getenv("GOOGLE_API_KEY")
grok_api_key       = os.getenv("GROK_API_KEY")
groq_api_key       = os.getenv("GROQ_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
hugging_face_token = os.getenv("HUGGING_FACE_TOKEN")


def _clean_key(k):
    """
    Clean an API key string by removing leading/trailing spaces
    and surrounding single or double quotes.

    Args:
        k: Raw key value from environment variables.

    Returns:
        str or None: Cleaned key if available, otherwise None.
    """
    if not k:
        return None
    return k.strip().strip('"').strip("'")


# Clean the Hugging Face token in case it includes extra quotes
hugging_face_token = _clean_key(hugging_face_token)

# Print a simple status message for each provider key
for name, key in [
    ("OpenAI",       openai_api_key),
    ("Anthropic",    anthropic_api_key),
    ("Google",       google_api_key),
    ("Grok",         grok_api_key),
    ("Groq",         groq_api_key),
    ("OpenRouter",   openrouter_api_key),
    ("HuggingFace",  hugging_face_token),
]:
    if key:
        print(f"{name} key exists and begins with '{key[:6]}'")
    else:
        print(f"{name} key not set (optional)")

## Initialize Clients for Multiple Model Providers

In this cell, we create client objects for the different model providers supported by the application.

The code initializes connections for:
- OpenAI
- Anthropic
- Google Gemini
- Grok
- Groq
- Ollama
- OpenRouter
- Hugging Face

Most providers are accessed using an OpenAI-compatible client interface with a custom `base_url`, while Hugging Face uses its own `InferenceClient`.

This design allows the app to switch between providers and models more easily while keeping the rest of the code consistent.

In [ ]:
# Initialize clients for different model providers

# Default OpenAI client
openai_client = OpenAI()

# Anthropic client through OpenAI-compatible interface
anthropic = OpenAI(
    api_key=anthropic_api_key,
    base_url="https://api.anthropic.com/v1/"
)

# Google Gemini client through OpenAI-compatible interface
gemini = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# Grok client through OpenAI-compatible interface
grok = OpenAI(
    api_key=grok_api_key,
    base_url="https://api.x.ai/v1"
)

# Groq client through OpenAI-compatible interface
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

# Local Ollama client
ollama = OpenAI(
    api_key=ollama_api_key,
    base_url="http://localhost:11434/v1"
)

# OpenRouter client
openrouter = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

# Hugging Face inference client
hf = InferenceClient(token=hugging_face_token) if hugging_face_token else None

## Configure Model Options and Compilation Commands

In this cell, we define configuration settings used by the application.

### Model Options
`MODEL_OPTIONS` lists the available models for each provider.  
This allows the Gradio interface to dynamically populate model selections based on the chosen provider.

### Client Mapping
`CLIENTS` maps each provider name to its corresponding API client object.  
This makes it easy to route requests to the correct provider.

### Compilation Commands
The application also defines commands used for compiling and running C++ code generated by the LLM.

- `compile_command` uses **clang++** with performance optimization flags.
- `run_command` executes the compiled program.

These commands are used in the **Python → C++ conversion tab** to compare execution performance.

In [ ]:
# Available model options for each provider
MODEL_OPTIONS = {
    "OpenAI":       ["gpt-5", "gpt-5.4-2026-03-05", "gpt-4.1-2025-04-14"],
    "Anthropic":    ["claude-sonnet-4-5-20250929", "claude-haiku-4-5"],
    "Gemini":       ["gemini-2.5-pro", "gemini-2.5-flash-lite"],
    "Grok":         ["grok-4", "grok-4-fast-non-reasoning"],
    "Groq":         ["openai/gpt-oss-120b"],
    "Ollama":       ["llama3.2", "qwen3.8b", "qwen3-coder:30b", "deepseek-r1:8b"],
    "OpenRouter":   ["qwen/qwen3-coder-plus"],
    "HuggingFace":  ["Qwen/Qwen2.5-14B-Instruct"],
}

# Mapping of provider names to their corresponding client objects
CLIENTS = {
    "OpenAI":       openai_client,
    "Anthropic":    anthropic,
    "Gemini":       gemini,
    "Grok":         grok,
    "Groq":         groq_client,
    "Ollama":       ollama,
    "OpenRouter":   openrouter,
}

# Command used to compile generated C++ code
compile_command = [
    "clang++",
    "-std=c++17",
    "-Ofast",
    "-mcpu=native",
    "-flto=thin",
    "-fvisibility=hidden",
    "-DNDEBUG",
    "main.cpp",
    "-o",
    "main",
]

# Command used to execute the compiled C++ binary
run_command = ["./main"]

## Update Model Selection and Route Requests to Providers

In this cell, we define helper functions for model selection and request routing.

### `update_model_dropdown()`
This function updates the Gradio model dropdown whenever a provider is selected.  
It looks up the available models for that provider and sets the first one as the default choice.

### `_call_model()`
This function sends chat-completion requests to the correct provider.

- If the provider is **Hugging Face**, it uses the Hugging Face inference client.
- For all other providers, it uses the matching OpenAI-compatible client.
- For GPT-family models, it optionally sets a higher `reasoning_effort` value.

These helper functions make the application dynamic and allow a single interface to work with multiple providers and model families.

In [ ]:
def update_model_dropdown(provider: str):
    """
    Update the model dropdown options based on the selected provider.

    Args:
        provider (str): Name of the selected provider.

    Returns:
        gr.Dropdown: A Gradio dropdown component populated with the
        models available for the selected provider.
    """
    choices = MODEL_OPTIONS.get(provider, [])
    return gr.Dropdown(choices=choices, value=choices[0] if choices else None)


def _call_model(provider: str, model: str, messages: list) -> str:
    """
    Route a chat-completion request to the correct provider client.

    This function handles two cases:
    - Hugging Face, which uses its own chat completion interface
    - all other providers, which are accessed through OpenAI-compatible clients

    Args:
        provider (str): Name of the model provider.
        model (str): Model name to use for the request.
        messages (list): List of chat messages in conversation format.

    Returns:
        str: The text content returned by the model.
    """
    # Handle Hugging Face separately because it uses a different client API
    if provider == "HuggingFace":
        if hf is None:
            return "Error: HuggingFace token not set."

        response = hf.chat_completion(
            model=model,
            messages=messages,
            max_tokens=4096,
        )
        return response.choices[0].message.content

    # Route request to the selected OpenAI-compatible provider client
    client = CLIENTS[provider]

    # Apply reasoning effort for GPT-family models when supported
    reasoning_effort = "high" if "gpt" in model.lower() else None

    kwargs = dict(model=model, messages=messages)
    if reasoning_effort:
        kwargs["reasoning_effort"] = reasoning_effort

    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

In [ ]:
system_prompt_convert = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def _user_prompt_convert(python: str) -> str:
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output.
Your response will be written to main.cpp and compiled with:
{compile_command}
Respond only with C++ code.

```python
{python}
```
"""

def port_to_cpp(provider: str, model: str, python: str) -> str:
    messages = [
        {"role": "system",  "content": system_prompt_convert},
        {"role": "user",    "content": _user_prompt_convert(python)},
    ]
    reply = _call_model(provider, model, messages)
    return reply.replace("```cpp", "").replace("```", "")


def run_python(code: str) -> str:
    buf = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buf
    try:
        exec(code, {"__builtins__": __builtins__})
        return buf.getvalue()
    except Exception as e:
        return f"Error: {e}"
    finally:
        sys.stdout = old_stdout


def compile_and_run(cpp_code: str) -> str:
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp_code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        return f"Compilation/run error:\n{e.stderr}"

In [ ]:
SUPPORTED_LANGUAGES = ["Python", "C++", "Rust", "JavaScript", "TypeScript", "Java"]

DOCSTRING_STYLE = {
    "Python":     "Google-style docstrings (Args:, Returns:, Raises:, Example:)",
    "C++":        "Doxygen-style comments (/// @brief, @param, @return)",
    "Rust":       "Rust doc comments (///, # Examples, # Panics, # Errors)",
    "JavaScript": "JSDoc comments (@param, @returns, @throws, @example)",
    "TypeScript": "JSDoc/TSDoc comments (@param, @returns, @throws, @example)",
    "Java":       "Javadoc comments (@param, @return, @throws, @since)",
}

system_prompt_docstring = """
You are an expert code documentation assistant.
Your task is to add clear, professional docstrings and inline comments to the provided code.
- Add module/file-level docstrings where appropriate.
- Add function/method docstrings using the style specified.
- Add class-level docstrings.
- Add concise inline comments for non-obvious logic.
- Do NOT change the code logic — only add documentation.
- Respond with the fully documented code only. No explanations outside the code.
"""

def _user_prompt_docstring(code: str, language: str) -> str:
    style = DOCSTRING_STYLE.get(language, "standard documentation comments")
    return f"""
Add comprehensive docstrings and comments to the following {language} code.
Use {style}.
Add:
  1. A module/file-level docstring at the top summarising what the code does.
  2. A docstring for every function, method, and class.
  3. Inline comments for any non-obvious logic.

Return the fully documented {language} code and nothing else.

```{language.lower()}
{code}
```
"""

def add_docstrings(provider: str, model: str, code: str, language: str) -> str:
    if not code.strip():
        return "Please paste some code first."
    messages = [
        {"role": "system", "content": system_prompt_docstring},
        {"role": "user",   "content": _user_prompt_docstring(code, language)},
    ]
    reply = _call_model(provider, model, messages)
    # Strip any markdown fences the model might have added
    for fence in [f"```{language.lower()}", "```python", "```cpp", "```rust",
                  "```javascript", "```typescript", "```java", "```"]:
        reply = reply.replace(fence, "")
    return reply.strip()

In [ ]:
# ── Sample code snippets ──────────────────────────────────────────────────────
PI_PYTHON = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
""".strip()

SAMPLE_UNDOCUMENTED = """
def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value

def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum
""".strip()

## Build and Launch the Gradio Application

This section creates the complete **Week 4 Gradio interface** and launches the application in the browser.

### Main Application Container
The app is created using `gr.Blocks()` and given the title **Code Tools — Week 4**.  
A short markdown heading is displayed at the top to describe the two tools included in the project.

### Tab 1: Python to C++ Converter
The first tab allows the user to:
- select a **provider**
- select a **model**
- paste or edit Python code
- convert it into optimized C++
- run the original Python code
- compile and run the generated C++ code

This tab includes:
- dropdowns for provider and model selection
- code editors for Python and C++
- buttons for running Python, converting to C++, and running C++
- output boxes for displaying execution results

The provider dropdown is connected to `update_model_dropdown()` so that the model list updates dynamically.

### Tab 2: Docstring Adder
The second tab allows the user to:
- choose a provider and model
- choose a programming language
- paste undocumented code
- generate a version with:
  - module docstrings
  - function docstrings
  - class docstrings
  - inline comments

This tab includes:
- provider, model, and language dropdowns
- input and output code editors
- a button to add docstrings and comments

The selected language also updates the input code editor so the syntax highlighting matches the chosen language.

### Event Handling
Each button is connected to a backend function using `.click()`:
- `port_to_cpp()` converts Python code to C++
- `run_python()` executes Python code
- `compile_and_run()` compiles and runs C++ code
- `add_docstrings()` documents the input code

### Launch
Finally, `ui.launch(inbrowser=True)` starts the application and opens it in the default browser.

Overall, this block connects all earlier helper functions into a complete interactive application with two practical LLM-powered coding tools.

In [ ]:
# Create the main Gradio application container.
# The title appears in the browser tab.
with gr.Blocks(title="Code Tools — Week 4") as ui:

    # Main heading shown at the top of the app
    gr.Markdown("# Code Tools — Week 4\nTwo tools: **Python to C++ converter** and **Docstring adder**.")

    # ── Shared model selector note ────────────────────────────────────────────
    # Each tab uses its own provider/model selection components
    # so both tools can work independently.

    # ── Tab 1: Python to C++ ──────────────────────────────────────────────────
    with gr.Tab("Python to C++"):

        # Short description for the first tool
        gr.Markdown("Convert Python to optimised C++ and optionally run both.")

        with gr.Row():
            # Dropdown to choose the model provider
            cpp_provider = gr.Dropdown(
                choices=list(MODEL_OPTIONS.keys()),
                value="OpenAI",
                label="Provider"
            )

            # Dropdown to choose the model based on the selected provider
            cpp_model = gr.Dropdown(
                choices=MODEL_OPTIONS["OpenAI"],
                value=MODEL_OPTIONS["OpenAI"][0],
                label="Model"
            )

        # Update model dropdown whenever the provider changes
        cpp_provider.change(fn=update_model_dropdown, inputs=[cpp_provider], outputs=[cpp_model])

        with gr.Row():
            # Left code editor for original Python code
            python_code = gr.Code(
                label="Python (original)",
                value=PI_PYTHON,
                language="python",
                lines=22
            )

            # Right code editor for generated C++ code
            cpp_code = gr.Code(
                label="C++ (generated)",
                value="",
                language="cpp",
                lines=22
            )

        with gr.Row():
            # Button to execute the Python code
            py_run_btn = gr.Button("▶ Run Python")

            # Button to convert Python code into C++
            convert_btn = gr.Button("⚡ Convert to C++")

            # Button to compile and run the generated C++ code
            cpp_run_btn = gr.Button("▶ Compile & Run C++")

        with gr.Row():
            # Output area for Python execution results
            py_out = gr.TextArea(label="Python output", lines=6)

            # Output area for C++ compilation/execution results
            cpp_out = gr.TextArea(label="C++ output", lines=6)

        # When convert button is clicked, generate C++ code from Python code
        convert_btn.click(
            fn=port_to_cpp,
            inputs=[cpp_provider, cpp_model, python_code],
            outputs=[cpp_code]
        )

        # When Run Python button is clicked, execute Python code and show output
        py_run_btn.click(
            fn=run_python,
            inputs=[python_code],
            outputs=[py_out]
        )

        # When Compile & Run C++ button is clicked, compile and run C++ code
        cpp_run_btn.click(
            fn=compile_and_run,
            inputs=[cpp_code],
            outputs=[cpp_out]
        )

    # ── Tab 2: Docstring Adder ────────────────────────────────────────────────
    with gr.Tab("Docstring Adder"):

        # Description for the second tool
        gr.Markdown(
            "Paste any code below and choose a language — the model will add "
            "module, function, class docstrings **and** inline comments without "
            "changing the logic."
        )

        with gr.Row():
            # Dropdown to choose the model provider
            doc_provider = gr.Dropdown(
                choices=list(MODEL_OPTIONS.keys()),
                value="OpenAI",
                label="Provider"
            )

            # Dropdown to choose the model
            doc_model = gr.Dropdown(
                choices=MODEL_OPTIONS["OpenAI"],
                value=MODEL_OPTIONS["OpenAI"][0],
                label="Model"
            )

            # Dropdown to choose the programming language of the input code
            doc_language = gr.Dropdown(
                choices=SUPPORTED_LANGUAGES,
                value="Python",
                label="Language"
            )

        # Update model dropdown whenever the provider changes
        doc_provider.change(fn=update_model_dropdown, inputs=[doc_provider], outputs=[doc_model])

        with gr.Row():
            # Input code editor for undocumented source code
            code_in = gr.Code(
                label="Original code (no docstrings)",
                value=SAMPLE_UNDOCUMENTED,
                language="python",
                lines=26
            )

            # Output code editor for documented code
            code_out = gr.Code(
                label="Documented code",
                value="",
                language="python",
                lines=26
            )

        # Button to trigger docstring and comment generation
        doc_btn = gr.Button("Add Docstrings & Comments")

        # When button is clicked, send code to the model and return documented code
        doc_btn.click(
            fn=add_docstrings,
            inputs=[doc_provider, doc_model, code_in, doc_language],
            outputs=[code_out]
        )

        # Update the code editor language hint when the user changes the language
        doc_language.change(
            fn=lambda lang: gr.Code(language=lang.lower()),
            inputs=[doc_language],
            outputs=[code_in]
        )

# Launch the Gradio application in the browser
ui.launch(inbrowser=True)